# M03 — CTA Data Quality Audit

## 1. Data load karo

In [1]:
import pandas as pd

bus = pd.read_csv(
    r"C:\UZZAM WORK\5th Semester\Artificial Intelligence & Expert Systems\Project\Transit_Ridership_M01_Setup\Transit_Ridership_M10_Reviewed\transit-ridership\backend\data\processed\bus_2022_2025.csv",
    dtype=str
)
rail = pd.read_csv(
    r"C:\UZZAM WORK\5th Semester\Artificial Intelligence & Expert Systems\Project\Transit_Ridership_M01_Setup\Transit_Ridership_M10_Reviewed\transit-ridership\backend\data\processed\rail_2022_2025.csv",
    dtype=str
)

bus = bus.apply(lambda column: column.str.strip().replace("", pd.NA))
rail = rail.apply(lambda column: column.str.strip().replace("", pd.NA))
datasets = {"Bus": bus, "Rail": rail}

for name, df in datasets.items():
    print(f"{name}: {len(df):,} rows, {len(df.columns)} columns")
    print("Columns:", list(df.columns))

Bus: 167,648 rows, 4 columns
Columns: ['route', 'date', 'daytype', 'rides']
Rail: 209,472 rows, 5 columns
Columns: ['station_id', 'stationname', 'date', 'daytype', 'rides']


## 2. Pehli 5 rows

In [2]:
bus.head()

,route,date,daytype,rides
0,1,01/03/2022,W,385
1,1,01/04/2022,W,470
2,1,01/05/2022,W,289
3,1,01/06/2022,W,339
4,1,01/07/2022,W,233


In [3]:
rail.head()

,station_id,stationname,date,daytype,rides
0,40010,Austin-Forest Park,01/01/2022,A,231
1,40020,Harlem-Lake,01/01/2022,A,500
2,40030,Pulaski-Lake,01/01/2022,A,284
3,40040,Quincy/Wells,01/01/2022,A,493
4,40050,Davis,01/01/2022,A,502


## 3. Missing values aur duplicate rows

In [4]:
for name, df in datasets.items():
    print(f"\n{name} — missing values")
    print(df.isna().sum().to_string())
    print("Exact duplicate extra rows:", df.duplicated().sum())


Bus — missing values
route      0
date       0
daytype    0
rides      0
Exact duplicate extra rows: 0

Rail — missing values
station_id     0
stationname    0
date           0
daytype        0
rides          0
Exact duplicate extra rows: 0


## 4. Dates aur route/station duplicates

In [5]:
for name, df in datasets.items():
    dates = pd.to_datetime(df["date"], format="mixed", errors="coerce")
    invalid_dates = df["date"].notna() & dates.isna()
    outside_range = dates.notna() & ~dates.between("2022-01-01", "2025-12-31 23:59:59.999999999")

    id_column = "route" if name == "Bus" else "station_id"
    keys = pd.DataFrame({"id": df[id_column], "date": dates.dt.normalize()})
    valid_keys = keys["id"].notna() & keys["date"].notna()

    print(f"\n{name} date range:", dates.min(), "to", dates.max())
    print("Invalid nonblank dates:", invalid_dates.sum())
    print("Dates outside 2022–2025:", outside_range.sum())
    print("Duplicate ID + date extra rows:", keys[valid_keys].duplicated().sum())
    print("Rows excluded from duplicate ID + date check:", (~valid_keys).sum())


Bus date range: 2022-01-01 00:00:00 to 2025-12-31 00:00:00
Invalid nonblank dates: 0
Dates outside 2022–2025: 0
Duplicate ID + date extra rows: 0
Rows excluded from duplicate ID + date check: 0

Rail date range: 2022-01-01 00:00:00 to 2025-12-31 00:00:00
Invalid nonblank dates: 0
Dates outside 2022–2025: 0
Duplicate ID + date extra rows: 0
Rows excluded from duplicate ID + date check: 0


## 5. Ridership aur day types

In [ ]:
for name, df in datasets.items():
    rides_text = df["rides"]
    has_commas = rides_text.str.fullmatch(r"\d{1,3}(?:,\d{3})+", na=False)
    rides_text = rides_text.where(~has_commas, rides_text.str.replace(",", "", regex=False))
    rides = pd.to_numeric(rides_text, errors="coerce")
    rides = rides.replace([float("inf"), float("-inf")], float("nan"))

    invalid_rides = df["rides"].notna() & rides.isna()
    unexpected_days = df["daytype"].notna() & ~df["daytype"].isin(["W", "A", "U"])
    # A->Saturday

    print(f"\n{name}")
    print("Invalid nonblank rides:", invalid_rides.sum())
    print("Negative rides:", (rides < 0).sum())
    print("Non-integer rides:", (rides.notna() & (rides % 1 != 0)).sum())
    print("Zero rides:", (rides == 0).sum())
    print("Unexpected nonblank day types:", unexpected_days.sum())
    print("Day type counts:")
    print(df["daytype"].value_counts(dropna=False).to_string())


Bus
Invalid nonblank rides: 0
Negative rides: 0
Non-integer rides: 0
Zero rides: 2
Unexpected nonblank day types: 0
Day type counts:
daytype
W    127701
U     20222
A     19725

Rail
Invalid nonblank rides: 0
Negative rides: 0
Non-integer rides: 0
Zero rides: 2742
Unexpected nonblank day types: 0
Day type counts:
daytype
W    146243
U     33264
A     29965


Zero rides automatically error nahi hoti. Data cleaning M04 mein hogi.